# Carbon Calculations for Emmission and Sequestration - single- site 

This Notebook outlines the estimates for Carbon Emmissions and Sequestrations given for the LivingWales LCCS habitat types. these estimates have been assessed through literature review which can be accessed here: <insert link>. 
    
These estimates are a guideline only as sequestration and emmission are highly influenced by other environmental variables such as soil type or seasonality. Therefopre this is taken as an annual rate average. 
    
This notebook is in development and if you have any input please contact us at livingwales-data@aber.ac.uk

# 0. Load Functions

In [ ]:
# Set up to import packages
import os
import sys

import datacube
import pandas as pd
import numpy as np
import geopandas as gpd

from datacube.utils.geometry import Geometry, CRS
from ipyleaflet import GeoData

import matplotlib.colors as colors
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

sys.path.append("../wales_utils/data_cube_utilities")
sys.path.append("../wales_utils/themes_utilities")
sys.path.append("../utils")

from display_tools import map_geom, rgb
from wdc_datahandling import geopolygon_masking

# Load in habitat specific functions (colour scheme etc.,)
import habitat

# 1. AREA  selection

Here you can choose the location of your site by: 
- Your own USER UPLOADS
- DRAW AND AREA
- View a model dataset for this notebook in CASE-STUDIES, or
- Choose a Welsh dataset

You may choose to add a buffer and add this to your site or subtract your site to visualise the buffer area only

## Load area selection functions

In [ ]:
import notebook_dropdowns

## 1.1 Select an area

In [ ]:
# initialization of site selection #
polygon_select = notebook_dropdowns.area_selection()

## 1.1.1 *OPTIONAL*  View dataset

In [ ]:
site_df = notebook_dropdowns.view_selected_polygon(polygon_select)
site_df#
# or just run "notebook_dropdowns.view_selected_polygon(polygon_select)"  to view only


## 1.1.2 *OPTIONAL* View site shape and area

In [ ]:
#  plot selected vector/polygon to confirm
notebook_dropdowns.plot_selected_polygon(polygon_select)


## 1.1.3 Select site on a map

Here you will be able to 
- Draw an area
- Select a site from a group, or
- Confirm your chosen site

In [ ]:
#  map to select or draw site
notebook_dropdowns.map_and_select_area(polygon_select)

## 1.1.4 *OPTIONAL* View selected or drawn dataset

In [ ]:
#  display what selected area is
selected_polygon = notebook_dropdowns.polygon_selected()
selected_polygon


## 1.1.5 CONFIRM site below 

In [ ]:
#  visualize selected option (add loading bar)
notebook_dropdowns.visualize_selected_area()

## 1.2 Would you like to add a buffer around your site? 

If you DO NOT require a buffer proceed to section 2.0

#### 1.2.1 Choose buffer distance

In [ ]:
# select buffer amount controls (you don't need to click confirm buffer button, once radio btn for option is clicked, it sets it)
notebook_dropdowns.include_buffer()

In [ ]:
# Confirm the active selected buffer amount is 
notebook_dropdowns.active_buffer()

### 1.2.2. Visualise Buffer + Site on a Map

In [ ]:
#  run the buffer and include selection method and visualize it 
notebook_dropdowns.buffer_include_selection()

### 1.2.3 Exclude site and retain BUFFER ONLY

In [ ]:
#  run the buffer and exclude selection method and visualize it 
notebook_dropdowns.buffer_exclude_selection()

### 1.2.4 CONFIRM site

In [ ]:
#  visualise what has been selected
notebook_dropdowns.visualize_selected_area()


## 1.3 Advanced Extras: 

- convert from Geopandas to GeoJson 
- convert from GeoJson to Geopandas

### 1.2.5 OPTIONAL Geopandas to GeoJson

In [ ]:
# if selected area data is in geopandas dataframe form, convert to GeoJson if needed 
geojson_format_selected_area = notebook_dropdowns.convert_to_geojson(selected_polygon)
geojson_format_selected_area

### 1.2.5 OPTIONAL GeoJson to Geopandas

In [ ]:
# if selected area data is in GeoJson  form, convert to  geopandas dataframe if needed
gpdf_format_selected_area = notebook_dropdowns.convert_to_geopandas_df(selected_polygon)
gpdf_format_selected_area

In [ ]:
selected_polygon

# 2. Analysis

In this section we extract an area from the Living Wales habitat map, calculate the area of each habitat and then convert these to carbon based on a look up table.

In [ ]:
# Select the year by editing this below.
year = "2020"

In [ ]:
geom = Geometry(geom=selected_polygon.iloc[0].geometry, crs=CRS("EPSG:27700"))

In [ ]:
query = {
    "geopolygon": geom,
    "time": (year + "-01-01", year + "-12-31"),
    "output_crs": "EPSG:27700",
    "resolution": (-10, 10),
    "dask_chunks": {"y": 2048, "x": 2048},
}

# Load habitat data for our polygon and time period
dc = datacube.Datacube()
habitat_dataset = dc.load(product="lw_habitats_lw", **query)
habitat_dataset_masked = geopolygon_masking(habitat_dataset, geopolygon=geom)

### 2.1 Habitat PLOT

In [ ]:
# Plotting
habitat_fig, ax =  plt.subplots(figsize=(10, 10))

habitat_plot = ax.imshow(
    habitat_dataset_masked.detailed.isel(time=0),
    cmap=habitat.get_detailedhabitat_cmap(),
    norm=habitat.get_detailedhabitat_norm(),
    extent=[
        habitat_dataset.x.min().data,
        habitat_dataset.x.max().data,
        habitat_dataset.y.min().data,
        habitat_dataset.y.max().data,
    ],
)

patches = [
    Patch(color=color, label=label[1]) for color, label in habitat.detailedhabitat_scheme.items()
]

ax.legend(handles=patches, bbox_to_anchor=(1.35, 1), facecolor="white")

# Add north arrow
x, y = -0.2, 1  # Adjust these values based on your plot
arrow_length = 0.1
ax.annotate('N', xy=(x, y), xytext=(x, y - arrow_length),
            arrowprops=dict(facecolor='black', width=5, headwidth=15),
            ha='center', va='center', fontsize=20,
            xycoords='axes fraction')

plt.show()

### 2.2 Habitats TABLE and Carbon Sequestration/Emmission values tables

In [ ]:
habitat_stats_df = habitat.stat_summary(habitat_dataset_masked.detailed, habitat.detailedhabitat_scheme)
# Rename catergory to 'Habitat Type' to match 
habitat_stats_df = habitat_stats_df.rename(columns={"CATEGORY" : "Habitat Type"})

In [ ]:
# Read in sequestration and emmision rates for habitats from CSV in shared space
carbon_data = pd.read_csv("/home/jovyan/shared_space/carbon_conversion/Habitat_Carbon_Sequestration_and_Emission_Data.csv")

In [ ]:
# Join tables to give carbon data for each habitat in the area
habitat_carbon_stats = pd.merge(habitat_stats_df, carbon_data, on="Habitat Type", how="left")

In [ ]:
# Multiply rates per ha with areas of each habitat.
habitat_carbon_stats["Lowest Sequestration Rate (tC/yr)"] = habitat_carbon_stats["Lowest Sequestration Rate (tC/ha/yr)"] * habitat_carbon_stats["HECTARE"]
habitat_carbon_stats["Highest Sequestration Rate (tC/yr)"] = habitat_carbon_stats["Highest Sequestration Rate (tC/ha/yr)"] * habitat_carbon_stats["HECTARE"]
habitat_carbon_stats["Lowest Emission Rate (tC/yr)"] = habitat_carbon_stats["Lowest Emission Rate (tC/ha/yr)"] * habitat_carbon_stats["HECTARE"]
habitat_carbon_stats["Highest Emission Rate (tC/yr)"] = habitat_carbon_stats["Highest Emission Rate (tC/ha/yr)"] * habitat_carbon_stats["HECTARE"]

In [ ]:
# Save out to a CSV for reading in excel and print table below
habitat_carbon_stats.to_csv(
    f"habitat_carbon_stats_{notebook_dropdowns.col_name_var}_{year}.csv", float_format="%.2f", index=False
)
habitat_carbon_stats

### 2.3 Habitats GRAPH 

(organise by greatest emmission to sequestration

TODO: Sections below this not currently working

In [ ]:
avg_emmision_rate = (habitat_carbon_stats["Lowest Emission Rate (tC/yr)"] + habitat_carbon_stats["Highest Emission Rate (tC/yr)"]) / 2
emission_lower_diff = habitat_carbon_stats["Lowest Emission Rate (tC/yr)"] - avg_emmision_rate
emission_upper_diff = habitat_carbon_stats["Highest Emission Rate (tC/yr)"] - avg_emmision_rate

In [ ]:
plt.bar(x=habitat_carbon_stats["Habitat Type"],
        height=habitat_carbon_stats["Highest Emission Rate (tC/yr)"],
        xerr=[emission_lower_diff, emission_upper_diff],
        label="Highest Emission Rate (tC/yr)")

### 2.4 Habitats STATISTICS

How much semi natural vs unnatural landscapes are there

## 3. Carbon emmissions 

Using the carbon emmisions table for LW Habitats

### 3.1 Emmissions PLOT x2 

Display heatmap of high vs low emmission habitats at the upper and lower range

### 3.2 Emissions TABLE

### 3.3 Emissions GRAPH

### 3.4 Emissions STATISTICS

## 4. Sequestration 

use carbon sequestration table 

### 4.1 Sequestration PLOT

Display heatmap of high vs low sequestration habitats at the upper and lower range

### 4.2 Sequestration TABLE

### 4.3 Sequestration GRAPH

### 4.4 Sequestration STATISTICS

## 5. The effect of Water  

To be added at a later date but attenuating the output to incllude the hydroperiod map 

### 5.1 Subheading PLOT

### 5.2 Subheading TABLE

### 5.3 Subheading GRAPH

### 5.4 Subheading STATISTICS